# 06 · k-Nearest Neighbours

k-NN is refreshingly simple: to predict a new point, find the *k* closest points
in the training data and let them vote (classification) or average (regression).
There's **no training phase** and **no optimizer** — the model just stores the
labelled data and does the work at prediction time.

Because everything hinges on distance, k-NN is sensitive to feature scale — a
place where the scaling from [`01_preprocessing`](01_preprocessing.ipynb)
matters. Our features here are already on a comparable scale.

## Data and a train/test split

k-NN predicts *new* points from *stored* ones, so we split into a labelled
training set and a held-out test set. Each split is its own `DolceSet`.

In [1]:
import numpy as np
import polars as pl

from dolcestat.preprocessing import DolceSet
from dolcestat.neighbors import KNNClassifier, KNNRegressor

rng = np.random.default_rng(0)
n = 300
x1 = rng.normal(0, 1, n)
x2 = rng.normal(0, 1, n)
prob = 1 / (1 + np.exp(-(1.5 * x1 - 2.0 * x2 + 0.4)))
label = (rng.uniform(size=n) < prob).astype(int)
df = pl.DataFrame({"x1": x1, "x2": x2, "label": label})

order = rng.permutation(n)
train_rows, test_rows = order[:240].tolist(), order[240:].tolist()

train = DolceSet()
train.load_from_polars_dataframe(df[train_rows], target_col="label")
test = DolceSet()
test.load_from_polars_dataframe(df[test_rows], target_col="label")

## Classification

Build a `KNNClassifier` from the training set and call `predict` on the test set.
You choose `k`, the distance `metric`, and how neighbours are `weights`-ed.
`predict` returns a classification **analyzer** carrying the predicted labels
(`y_fit`) and per-class probabilities (`proba`).

In [2]:
knn = KNNClassifier(train)
result = knn.predict(test, k=7, metric="euclidean", weights="distance")

print("predicted labels:", result.y_fit[:10])
print("class probabilities (first 3 rows):")
print(np.round(result.proba[:3], 2))

predicted labels: [0 1 0 1 0 0 1 0 0 0]
class probabilities (first 3 rows):
[[0.51 0.49]
 [0.08 0.92]
 [1.   0.  ]]


## The knobs

- **`k`** — how many neighbours vote. Small `k` fits tightly (low bias, high
  variance); large `k` smooths.
- **`metric`** — `"euclidean"`, `"manhattan"`, or `"minkowski"` (with
  `minkowski_p`).
- **`weights`** — `"uniform"` (every neighbour equal) or distance-based
  (`"distance"`, `"squared_distance"`, `"gaussian"`), so closer neighbours count
  for more.

In [3]:
alt = knn.predict(test, k=15, metric="manhattan", weights="uniform")
print("k=15, manhattan, uniform — first 10 labels:", alt.y_fit[:10])

k=15, manhattan, uniform — first 10 labels: [1 1 0 1 0 0 1 0 1 0]


## Regression

`KNNRegressor` works the same way but **averages** the neighbours' target values
(weighted, if you ask). Here's a quick regression example on its own data.

In [4]:
rng = np.random.default_rng(1)
m = 200
a = rng.normal(0, 1, m)
b = rng.normal(0, 1, m)
target = 2.0 * a - 1.0 * b + 0.5 + rng.normal(0, 0.3, m)
rdf = pl.DataFrame({"a": a, "b": b, "target": target})

ro = rng.permutation(m)
rtrain = DolceSet()
rtrain.load_from_polars_dataframe(rdf[ro[:160].tolist()], target_col="target")
rtest = DolceSet()
rtest.load_from_polars_dataframe(rdf[ro[160:].tolist()], target_col="target")

reg_result = KNNRegressor(rtrain).predict(rtest, k=10, weights="distance")
print("predicted values:", np.round(reg_result.y_fit[:6], 2))

predicted values: [ 1.47 -2.47  1.62 -2.05  2.58  1.61]


## Recap

No optimizer, no weights to learn — just neighbours and a distance. Both
`predict` calls handed back analyzers; in [`07_metrics`](07_metrics.ipynb) we
finally measure how good every model in this series actually is.